# Visualização de Features de uma CNN (ResNet50)

Neste notebook abrimos a "caixa" de uma rede convolucional já treinada:
vamos olhar os **kernels** da primeira camada e os **feature maps**
produzidos por uma camada intermediária.

**Atenção**: *Material complementar para estudo individual. Não se preocupe em compreender os detalhes neste momento. Trabalharemos eles em futuras aulas.*

## [1] Importação de bibliotecas

In [ ]:
"""
Define uma seed global para garantir reprodutibilidade máxima
no treinamento com TensorFlow/Keras.

NOTE 1: Para aulas, deixo essa parte comentada para demonstrar o
        comportamento estocástico das práticas com DL.
NOTE 2: Na prática diária com DL, determinamos as SEEDs somente no
        split dos dados, pois operações em GPU sofrem perda de
        performance quando forçadas ao determinismo.
NOTE 3: Em alguns casos de treinamento avançado com DL, não é
        possível estabelecer 100% de determinismo devido à alta
        dependência de operações paralelizadas.
"""
# 1. Variáveis de ambiente (precisam ser definidas ANTES de importar TF!
import os
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'      # força kernels determinísticos
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'    # determinismo em GPU (cuDNN)


import tensorflow as tf
import keras
import numpy as np
import cv2
import matplotlib.pyplot as plt
from absl import logging

# seeds determinadas após importação dos módulos
keras.utils.set_random_seed(SEED)
# 3. Habilita operações determinísticas no TF (TF >= 2.8)
# NOTE: Haverá perda de performance
# NOTE: Em alguns casos, ainda não será 100% determinístico (paralelismo!)
tf.config.experimental.enable_op_determinism()

# silenciar alguns avisos do tf que não serão relevantes
tf.get_logger().setLevel('ERROR')
logging.set_verbosity(logging.ERROR)

## Funções auxiliares

Execute esta célula uma vez, no início da sessão. Ela define as funções de
apoio usadas ao longo do notebook — não é preciso importar nada além do que
já foi importado acima.

In [ ]:
def load_image(path, size=(256, 256)):
    img = cv2.imread(path)
    img = cv2.resize(img, size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


def plot_images(images, scale=1.0):
    """
    Plota uma lista de imagens.
    """
    num_images = len(images)
    fig, axes = plt.subplots(1, num_images, figsize=(scale * num_images, scale))
    if num_images == 1:
        axes = [axes]
    for ax, img in zip(axes, images):
        ax.imshow(img, cmap='gray')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Imagem de entrada

Envie uma imagem do seu computador. Se você não enviar nada, uma imagem de
exemplo é baixada automaticamente. O arquivo é salvo em `data/examples/3.jpg`.

In [ ]:
import os
import shutil

os.makedirs('data/examples', exist_ok=True)
image_path = 'data/examples/3.jpg'

try:
    from google.colab import files
    print('Selecione uma imagem (ou cancele para usar a imagem de exemplo):')
    uploaded = files.upload()
except Exception:
    uploaded = {}

if uploaded:
    nome = next(iter(uploaded))
    shutil.move(nome, image_path)
    print(f'Imagem enviada salva em: {image_path}')
elif not os.path.exists(image_path):
    !wget -q -O {image_path} https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg
    print(f'Imagem de exemplo baixada em: {image_path}')

## [2] Carregamento do modelo pré-treinado (ResNet50)

Carregamos a ResNet50 já treinada no ImageNet, sem as camadas finais
de classificação (`include_top=False`), pois estamos interessados apenas
nas representações internas (features) aprendidas pela rede.

In [ ]:
resnet50 = keras.applications.ResNet50(
    weights='imagenet',
    input_shape=(224, 224, 3),
    include_top=False
)

# Exibe a arquitetura completa do modelo (útil para identificar nomes de camadas)
resnet50.summary()

## [3] Criação do extrator de features

Criamos um novo modelo que reaproveita os pesos da ResNet50, mas cuja
saída é a ativação de uma camada intermediária específica.
Isso nos permite "espiar" o que a rede está enxergando naquele ponto.

In [ ]:
target_layer = 'conv1_relu'

output_layer = resnet50.get_layer(target_layer).output
feature_extractor = keras.Model(inputs=resnet50.input, outputs=output_layer)

## [4] Função de pré-processamento de imagem

Função auxiliar que carrega uma imagem do disco e a converte para o
formato esperado pela ResNet50 (tamanho, tipo e normalização corretos).

In [ ]:
def preprocess_image(image_path):
    """
    Carrega uma imagem e aplica o pré-processamento exigido pela ResNet50.

    Parâmetros
    ----------
    image_path : str
        Caminho para o arquivo de imagem.

    Retorna
    -------
    np.ndarray
        Array pronto para ser usado como entrada da rede (shape: 1,224,224,3).
    """
    img = keras.utils.load_img(image_path, target_size=(224, 224))
    img_array = keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # adiciona dimensão de batch
    img_array = keras.applications.resnet50.preprocess_input(img_array)
    return img_array

## [5] Carregamento e pré-processamento da imagem de entrada

In [ ]:
# Versão "crua" da imagem, apenas para exibição visual
original_image = load_image(image_path)

# Versão pré-processada (input tensor), pronta para entrar na rede neural
input_image = preprocess_image(image_path)

## [6] Extração das features da imagem

Passamos a imagem pela rede até a camada escolhida e obtemos os
mapas de ativação (feature maps) gerados naquele ponto.

In [ ]:
feature_maps = feature_extractor.predict(input_image)

print(f'Shape dos feature maps extraídos: {feature_maps.shape}')

## [7] Função para visualizar kernels (filtros) da primeira camada

Os kernels da primeira camada convolucional são os únicos que podem
ser interpretados diretamente como "imagens", pois operam sobre os
3 canais RGB da entrada.

In [ ]:
def plot_kernels(num_kernels, scale=1):
    """
    Seleciona e exibe aleatoriamente alguns kernels da primeira camada
    convolucional (conv1_conv) da ResNet50.

    NOTA: Somente os kernels da primeira camada podem ser interpretados como
    imagens inteligíveis. Da segunda em diante, trabalham no espaço hiperdimensional

    Parâmetros
    ----------
    num_kernels : int
        Quantidade de kernels a serem exibidos.
    scale : float
        Fator de escala aplicado na visualização.
    """
    layer_kernels = resnet50.get_layer('conv1_conv').weights[0].numpy()

    # Sorteia índices de kernels sem repetição
    kernels_idx = np.random.choice(layer_kernels.shape[-1], num_kernels, replace=False)
    kernels = layer_kernels[:, :, :, kernels_idx]

    # Normaliza para o intervalo de imagem (0-255) e reorganiza os eixos
    kernels = (kernels * 255).astype('uint8').transpose((-1, 0, 1, 2))

    plot_images(kernels, scale=4)

## [8] Função para visualizar feature maps de uma camada intermediária

Diferente dos kernels da primeira camada, os feature maps podem ser
interpretados como imagens cinza (único canal), revelando quais regiões
da imagem ativam mais a rede.

In [ ]:
def plot_feature_maps(feature_maps, num_maps=5, scale=1):
    """
    Seleciona e exibe aleatoriamente alguns feature maps 2D gerados
    pela camada intermediária escolhida.

    Parâmetros
    ----------
    feature_maps : np.ndarray
        Saída da camada intermediária (shape: 1, H, W, C).
    num_maps : int
        Quantidade de feature maps a serem exibidos.
    scale : float
        Fator de escala aplicado na visualização.
    """
    maps_idx = np.random.choice(range(feature_maps.shape[-1]), num_maps, replace=False)
    selected_feature_maps = feature_maps[0, :, :, maps_idx]

    plot_images(selected_feature_maps, scale)

## [9] Visualização: imagem original

In [ ]:
plot_images([original_image], scale=4)

## [10] Visualização: kernels da primeira camada convolucional

- Execute várias vezes para observar diferentes kernels aleatórios da mesma camada

In [ ]:
plot_kernels(num_kernels=5)

## [11] Visualização: feature maps da camada intermediária escolhida

- Execute várias vezes para observar diferentes feature maps aleatórios da mesma camada

In [ ]:
plot_feature_maps(feature_maps, num_maps=2, scale=3)